# Notebook 3 · Évaluation (J3) · Étape 5, puis préparation du déploiement (étape 6)

1. Choisir le **seuil de décision** en validation croisée sur l'entraînement.
2. Évaluer **une seule fois** sur le test : matrice de confusion, rappel, F1, AUC.
3. Analyser où le modèle se trompe.
4. Exporter `model.joblib` et `requirements.txt` pour l'application Streamlit.

In [ ]:
# Imports (toutes ces bibliothèques sont déjà installées dans Colab)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 0. Mise en place (identique dans les trois notebooks)

In [ ]:
# ===================== CONFIG : identique dans les 3 notebooks =====================
# Si vous modifiez une valeur ici, modifiez-la aussi dans les deux autres notebooks.
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
LOCAL_PATH = "online_shoppers_intention.csv"   # utilisé si le fichier a été téléversé dans Colab

TARGET = "Revenue"
NUM_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "SpecialDay",
]
LEAKY_COL = "PageValues"   # calculée à partir des transactions : testée avec et sans
CAT_COLS = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
BOOL_COLS = ["Weekend"]

FEATURE_SETS = {
    "sans_PageValues": NUM_COLS + CAT_COLS + BOOL_COLS,
    "avec_PageValues": NUM_COLS + [LEAKY_COL] + CAT_COLS + BOOL_COLS,
}

# Garde-fou : les 3 notebooks doivent obtenir exactement le même découpage.
EXPECTED_SPLIT = (9764, 2441)   # (lignes d'entraînement, lignes de test)

In [ ]:
# ===================== FONCTIONS COMMUNES : identiques dans les 3 notebooks =====================
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def load_raw():
    """Étape 1 : charge le CSV officiel (fichier local, sinon page UCI, sinon téléversement)."""
    if Path(LOCAL_PATH).exists():
        return pd.read_csv(LOCAL_PATH)
    try:
        return pd.read_csv(DATA_URL)
    except Exception as err:
        print("Téléchargement impossible :", err, "-> téléversez online_shoppers_intention.csv")
        from google.colab import files
        files.upload()
        return pd.read_csv(LOCAL_PATH)


def clean(df):
    """Nettoyage sans apprentissage (aucune statistique calculée) : sûr avant le découpage."""
    df = df.copy()
    df["Month"] = df["Month"].replace({"June": "Jun"})   # seul mois écrit en entier
    df["Weekend"] = df["Weekend"].astype(int)
    df[TARGET] = df[TARGET].astype(int)
    for col in CAT_COLS:                                 # les codes sont des catégories
        df[col] = df[col].astype(str)
    return df.drop_duplicates().reset_index(drop=True)   # avant le découpage, sinon fuite


def split(df):
    """Découpage stratifié, toujours avec la même graine."""
    X = df.drop(columns=TARGET)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    assert (len(X_train), len(X_test)) == EXPECTED_SPLIT, "Découpage différent des autres notebooks !"
    return X_train, X_test, y_train, y_test


def make_preprocessor(feature_set):
    """Étape 2 : à n'ajuster que sur l'entraînement (ou dans un Pipeline)."""
    num = [c for c in FEATURE_SETS[feature_set] if c in NUM_COLS + [LEAKY_COL]]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("bool", "passthrough", BOOL_COLS),
    ], remainder="drop")   # remainder="drop" retire PageValues dans la variante « sans »


def build_estimator(name, params=None):
    """Étape 3 : les modèles candidats."""
    estimator = {
        "Reference_naive": DummyClassifier(strategy="most_frequent"),
        "Regression_logistique": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Foret_aleatoire": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient_boosting": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE),
    }[name]
    if params:
        estimator.set_params(**params)
    return estimator


def make_pipeline(name, feature_set, params=None):
    return Pipeline([("prep", make_preprocessor(feature_set)),
                     ("model", build_estimator(name, params))])


CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
df = clean(load_raw())
X_train, X_test, y_train, y_test = split(df)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape,
      "| taux d'achat train / test :", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

### Configuration retenue en J2

Remplacez ce dictionnaire par la sortie de la cellule 2.4 du notebook 2.

In [ ]:
BEST = {
    "sans_PageValues": {"model": "Gradient_boosting", "params": {"learning_rate": 0.05, "max_depth": None}, "cv_f1": 0.431},
    "avec_PageValues": {"model": "Foret_aleatoire", "params": {"max_depth": None, "min_samples_leaf": 1}, "cv_f1": 0.685},
}

DEPLOYED = "sans_PageValues"   # la seule variante entièrement connue pendant la visite

# 1. Choix du seuil en validation croisée (sans toucher au test)

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict


def choose_threshold(feature_set):
    cfg = BEST[feature_set]
    pipe = make_pipeline(cfg["model"], feature_set, cfg["params"])
    proba = cross_val_predict(pipe, X_train, y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_train, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    i = int(np.argmax(f1[:-1]))
    return float(thresholds[i]), precision, recall, f1


THRESHOLDS = {}
for feature_set in FEATURE_SETS:
    thr, precision, recall, f1 = choose_threshold(feature_set)
    THRESHOLDS[feature_set] = thr
    print(f"{feature_set:16s} seuil = {thr:.3f} | F1 CV au seuil = {f1.max():.3f}")
    plt.plot(recall, precision, label=feature_set)
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.title("Courbe précision-rappel (validation croisée sur le train)")
plt.legend()
plt.show()

# 2. Évaluation unique sur le test

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
                             f1_score, recall_score, roc_auc_score)

FINAL = {}
metrics = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, feature_set in zip(axes, FEATURE_SETS):
    cfg = BEST[feature_set]
    pipe = make_pipeline(cfg["model"], feature_set, cfg["params"]).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLDS[feature_set]).astype(int)
    FINAL[feature_set] = {"pipe": pipe, "y_pred": y_pred, "proba": proba}
    metrics.append({"variables": feature_set, "modele": cfg["model"],
                    "seuil": round(THRESHOLDS[feature_set], 3),
                    "f1": round(f1_score(y_test, y_pred), 3),
                    "rappel": round(recall_score(y_test, y_pred), 3),
                    "auc": round(roc_auc_score(y_test, proba), 3)})
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, display_labels=["Pas d'achat", "Achat"])
    ax.set_title(feature_set)
plt.tight_layout()
plt.show()

metrics = pd.DataFrame(metrics)
display(metrics)
print(classification_report(y_test, FINAL[DEPLOYED]["y_pred"], target_names=["Pas d'achat", "Achat"]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for feature_set in FEATURE_SETS:
    RocCurveDisplay.from_predictions(y_test, FINAL[feature_set]["proba"], name=feature_set, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("Courbes ROC sur le test")
plt.show()

# 3. Où le modèle se trompe-t-il ?

Rappel (part des acheteurs détectés) par groupe, pour le modèle déployé.

In [ ]:
errors = X_test.assign(achat=y_test.values, detecte=FINAL[DEPLOYED]["y_pred"])
buyers = errors[errors.achat == 1]
for col in ["VisitorType", "Month"]:
    table = buyers.groupby(col)["detecte"].agg(rappel="mean", acheteurs="size").round(3)
    display(table.sort_values("rappel"))

### Limites (à rédiger, puis reporter dans la section 4 du README)

- *Erreurs les plus fréquentes :* …
- *Groupes mal servis :* …
- *Usages à éviter :* …

# 4. Export pour l'application (étape 6)

In [ ]:
import joblib

deployed_cols = FEATURE_SETS[DEPLOYED]
num_cols = [c for c in deployed_cols if c in NUM_COLS + [LEAKY_COL]]
bundle = {
    "pipeline": FINAL[DEPLOYED]["pipe"],
    "threshold": THRESHOLDS[DEPLOYED],
    "feature_set": DEPLOYED,
    "features": deployed_cols,
    "num_cols": num_cols,
    "cat_cols": CAT_COLS,
    "bool_cols": BOOL_COLS,
    # bornes vues à l'entraînement : l'application refuse ce qui sort de ce domaine
    "schema": {
        "num": {c: [float(X_train[c].min()), float(X_train[c].max())] for c in num_cols},
        "cat": {c: sorted(X_train[c].unique().tolist()) for c in CAT_COLS},
    },
    "metrics_test": metrics[metrics.variables == DEPLOYED].iloc[0].to_dict(),
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
joblib.dump(bundle, "model.joblib")

# requirements.txt avec les versions EXACTES de cet environnement (évite les erreurs de chargement)
Path("requirements.txt").write_text(
    "streamlit\n"
    f"scikit-learn=={sklearn.__version__}\n"
    f"pandas=={pd.__version__}\n"
    f"numpy=={np.__version__}\n"
    f"joblib=={joblib.__version__}\n"
)
print(Path("requirements.txt").read_text())
print("Modèle exporté :", bundle["metrics_test"])

try:
    from google.colab import files
    files.download("model.joblib")
    files.download("requirements.txt")
except ImportError:
    print("Hors Colab : fichiers écrits dans", Path.cwd())

## Bilan J3 (à rédiger)

- *Modèle déployé, seuil et scores sur le test :* …
- *Pourquoi la variante sans `PageValues` :* …
- Copiez `model.joblib` et `requirements.txt` dans le dossier `app/`, puis poussez-le sur GitHub.